<div dir="rtl">

# ۰۱ — اولین سرور MCP و دیدن پیام‌های خام JSON-RPC

در اسلایدها دیدیم که MCP روی **JSON-RPC 2.0** ساخته شده و ارتباط کلاینت و سرور یک **چرخه‌ی حیات** دارد:

۱. راه‌اندازی اولیه (`initialize`) ← ۲. تبادل پیام ← ۳. پایان ارتباط

در این نوت‌بوک **بدون هیچ کلاینت MCP**، فقط با درخواست HTTP معمولی (کتابخانه‌ی `httpx`)، همین مراحل را قدم‌به‌قدم انجام می‌دهیم و پیام‌ها را با چشم خودمان می‌بینیم.

> هدف این نیست که در عمل این‌طور کد بنویسید؛ در نوت‌بوک بعد SDK همه‌ی این کارها را برایمان انجام می‌دهد. هدف این است که بدانیم SDK پشت صحنه چه می‌کند.

</div>

In [9]:
from dotenv import load_dotenv
load_dotenv()  # reads OPENAI_API_KEY (and other settings) from the .env file

from mcp_helpers import setup_notebook, start_server, stop_server, parse_sse, show_json
setup_notebook()

Notebook ready | Python 3.12.12 | win32


<div dir="rtl">

## ساخت اولین سرور

برای ساخت سرور از `FastMCP` استفاده می‌کنیم. این کلاس، رابط سطح بالای SDK رسمی MCP است:

- با دکوریتور `@mcp.tool()` یک تابع معمولی پایتون به **ابزار MCP** تبدیل می‌شود.
- **نام تابع** همان نام ابزار است.
- **docstring** همان توضیح ابزار است. مدل زبانی بعداً همین متن را می‌خواند تا تصمیم بگیرد کی از ابزار استفاده کند.
- **type hint**ها (`int`، `float`) به‌صورت خودکار به JSON Schema تبدیل می‌شوند تا کلاینت بداند چه ورودی‌هایی بفرستد.

دستور `%%writefile` کد سلول را اجرا نمی‌کند؛ فقط آن را در فایل `servers/demo_server.py` ذخیره می‌کند.

</div>

In [ ]:
import os
os.makedirs("servers", exist_ok=True)

In [ ]:
%%writefile servers/demo_server.py
from mcp.server.fastmcp import FastMCP

# The name is what clients see in "serverInfo"
mcp = FastMCP("Demo Server", port=8000)


@mcp.tool()
def add(a: int, b: int) -> int:
    """Add two integers and return the sum."""
    return a + b


@mcp.tool()
def divide(a: float, b: float) -> float:
    """Divide a by b."""
    return a / b


if __name__ == "__main__":
    # Streamable HTTP: the server listens on http://127.0.0.1:8000/mcp
    mcp.run(transport="streamable-http")


<div dir="rtl">

## اجرای سرور

سرور یک برنامه‌ی جداست و باید در یک پروسه‌ی جدا اجرا شود. دو راه دارید:

**راه ۱: ترمینال جدا.** در پوشه‌ی نوت‌بوک‌ها و با محیط مجازی فعال:

- ویندوز: `python servers\demo_server.py`
- لینوکس: `python3 servers/demo_server.py`

**راه ۲: از داخل نوت‌بوک.** تابع `start_server` همین دستور را در پس‌زمینه اجرا می‌کند و صبر می‌کند تا پورت باز شود. در این دوره از این راه استفاده می‌کنیم.

> اگر از راه ۱ استفاده کردید، سلول زیر را اجرا نکنید.

</div>

In [13]:
server = start_server("servers/demo_server.py", port=8000)

Server started: servers/demo_server.py   (port 8000, pid 3288, log: logs\demo_server_8000.log)


<div dir="rtl">

## یک تابع کوچک برای ارسال پیام

در **Streamable HTTP** همه‌ی پیام‌های کلاینت با `POST` به یک آدرس واحد (`/mcp`) فرستاده می‌شوند. دو هدر لازم است:

- `Content-Type: application/json`: بدنه‌ی درخواست JSON است.
- `Accept: application/json, text/event-stream`: کلاینت می‌گوید هم پاسخ JSON ساده و هم پاسخ به‌شکل **SSE** را می‌پذیرد.

تابع `send` پیام را می‌فرستد، چیزی را که فرستاده و چیزی را که دریافت کرده چاپ می‌کند، و پاسخ HTTP را برمی‌گرداند.

</div>

In [17]:
import json
import httpx

URL = "http://127.0.0.1:8000/mcp"
BASE_HEADERS = {
    "Content-Type": "application/json",
    "Accept": "application/json, text/event-stream",
}


def send(message, session_id=None, protocol_version=None):
    headers = dict(BASE_HEADERS)
    if session_id:
        headers["Mcp-Session-Id"] = session_id
    if protocol_version:
        headers["MCP-Protocol-Version"] = protocol_version

    print(">>> REQUEST")
    print(json.dumps(message, indent=2, ensure_ascii=False))
    response = httpx.post(URL, headers=headers, json=message)
    print(f"\n<<< RESPONSE  (HTTP {response.status_code}, content-type: {response.headers.get('content-type')})")
    print(response.text if response.text else "(empty body)")
    return response

<div dir="rtl">

## گام ۱: راه‌اندازی اولیه (`initialize`)

اولین پیام همیشه `initialize` است. کلاینت در این پیام سه چیز را اعلام می‌کند:

- `protocolVersion`: نسخه‌ی پروتکلی که کلاینت می‌شناسد.
- `capabilities`: قابلیت‌های کلاینت. اینجا خالی است.
- `clientInfo`: نام و نسخه‌ی کلاینت.

دقت کنید که پیام `id` دارد، پس یک **درخواست** است و منتظر پاسخ می‌مانیم.

</div>

In [19]:
init_request = {
    "jsonrpc": "2.0",
    "id": 1,
    "method": "initialize",
    "params": {
        "protocolVersion": "2025-11-25",
        "capabilities": {},
        "clientInfo": {"name": "raw-httpx-client", "version": "1.0"},
    },
}
r = send(init_request)

>>> REQUEST
{
  "jsonrpc": "2.0",
  "id": 1,
  "method": "initialize",
  "params": {
    "protocolVersion": "2025-11-25",
    "capabilities": {},
    "clientInfo": {
      "name": "raw-httpx-client",
      "version": "1.0"
    }
  }
}

<<< RESPONSE  (HTTP 200, content-type: text/event-stream)
event: message
data: {"jsonrpc":"2.0","id":1,"result":{"protocolVersion":"2025-11-25","capabilities":{"experimental":{},"prompts":{"listChanged":false},"resources":{"subscribe":false,"listChanged":false},"tools":{"listChanged":false}},"serverInfo":{"name":"Demo Server","version":"1.25.0"}}}




<div dir="rtl">

دو نکته در پاسخ:

۱. **بدنه به‌شکل SSE است**، نه JSON خالص: یک خط `event: message` و یک خط `data: {...}`. سرور حق دارد پاسخ را به‌شکل یک «جریان رویداد» بفرستد تا اگر لازم شد، قبل از پاسخ نهایی پیام‌های دیگری (مثلاً گزارش پیشرفت) هم بفرستد.

۲. **سرور یک شناسه‌ی session ساخته** و آن را در هدر `Mcp-Session-Id` برگردانده است. از این به بعد باید این شناسه را در همه‌ی درخواست‌ها بفرستیم.

بیایید پاسخ را از قالب SSE بیرون بکشیم:

</div>

In [29]:
r.headers["mcp-session-id"]

'1fa9b8b7c8e847eebd66bf050295d601'

In [31]:
session_id = r.headers["mcp-session-id"]
print("Session ID:", session_id)

init_result = parse_sse(r.text)[0]["result"]
show_json(init_result)

Session ID: 1fa9b8b7c8e847eebd66bf050295d601
{
  "protocolVersion": "2025-11-25",
  "capabilities": {
    "experimental": {},
    "prompts": {
      "listChanged": false
    },
    "resources": {
      "subscribe": false,
      "listChanged": false
    },
    "tools": {
      "listChanged": false
    }
  },
  "serverInfo": {
    "name": "Demo Server",
    "version": "1.25.0"
  }
}


<div dir="rtl">

سرور در پاسخ گفته است:

- `protocolVersion`: نسخه‌ای که هر دو طرف روی آن توافق کردند.
- `capabilities`: سرور چه چیزهایی دارد: `tools`، `resources` و `prompts`.
- `serverInfo`: نام سرور، همان `"Demo Server"`.

## گام ۲: اعلام آمادگی (`notifications/initialized`)

کلاینت حالا به سرور می‌گوید «پاسخت را گرفتم، آماده‌ام». این پیام **`id` ندارد**. طبق JSON-RPC، پیام بدون `id` یک **Notification** است: «بفرست و فراموش کن»، و سرور هیچ پاسخی برنمی‌گرداند.

</div>

In [35]:
PROTOCOL = init_result["protocolVersion"]

r = send(
    {"jsonrpc": "2.0", "method": "notifications/initialized"},
    session_id=session_id,
    protocol_version=PROTOCOL,
)

>>> REQUEST
{
  "jsonrpc": "2.0",
  "method": "notifications/initialized"
}

<<< RESPONSE  (HTTP 202, content-type: application/json)
(empty body)


<div dir="rtl">

وضعیت HTTP برابر `202 Accepted` است و بدنه خالی است: سرور پیام را پذیرفت، ولی چون Notification بود پاسخی نفرستاد.

## گام ۳: کشف ابزارها (`tools/list`)

حالا ارتباط برقرار است. اول می‌پرسیم سرور چه ابزارهایی دارد:

</div>

In [37]:
r = send({"jsonrpc": "2.0", "id": 2, "method": "tools/list"}, session_id=session_id, protocol_version=PROTOCOL)

>>> REQUEST
{
  "jsonrpc": "2.0",
  "id": 2,
  "method": "tools/list"
}

<<< RESPONSE  (HTTP 200, content-type: text/event-stream)
event: message
data: {"jsonrpc":"2.0","id":2,"result":{"tools":[{"name":"add","description":"Add two integers and return the sum.","inputSchema":{"properties":{"a":{"title":"A","type":"integer"},"b":{"title":"B","type":"integer"}},"required":["a","b"],"title":"addArguments","type":"object"},"outputSchema":{"properties":{"result":{"title":"Result","type":"integer"}},"required":["result"],"title":"addOutput","type":"object"}},{"name":"divide","description":"Divide a by b.","inputSchema":{"properties":{"a":{"title":"A","type":"number"},"b":{"title":"B","type":"number"}},"required":["a","b"],"title":"divideArguments","type":"object"},"outputSchema":{"properties":{"result":{"title":"Result","type":"number"}},"required":["result"],"title":"divideOutput","type":"object"}}]}}




In [39]:
for tool in parse_sse(r.text)[0]["result"]["tools"]:
    print("name:        ", tool["name"])
    print("description: ", tool["description"])
    print("inputSchema: ", json.dumps(tool["inputSchema"], ensure_ascii=False))
    print("outputSchema:", json.dumps(tool.get("outputSchema"), ensure_ascii=False))
    print()

name:         add
description:  Add two integers and return the sum.
inputSchema:  {"properties": {"a": {"title": "A", "type": "integer"}, "b": {"title": "B", "type": "integer"}}, "required": ["a", "b"], "title": "addArguments", "type": "object"}
outputSchema: {"properties": {"result": {"title": "Result", "type": "integer"}}, "required": ["result"], "title": "addOutput", "type": "object"}

name:         divide
description:  Divide a by b.
inputSchema:  {"properties": {"a": {"title": "A", "type": "number"}, "b": {"title": "B", "type": "number"}}, "required": ["a", "b"], "title": "divideArguments", "type": "object"}
outputSchema: {"properties": {"result": {"title": "Result", "type": "number"}}, "required": ["result"], "title": "divideOutput", "type": "object"}



<div dir="rtl">

همه‌ی این اطلاعات از روی خود تابع پایتون ساخته شده است:

| در کد پایتون | در پاسخ `tools/list` |
|---|---|
| نام تابع `add` | `name` |
| docstring | `description` |
| `a: int, b: int` | `inputSchema` (یک JSON Schema) |
| `-> int` | `outputSchema` |

این همان اطلاعاتی است که بعداً به مدل زبانی داده می‌شود.

## گام ۴: فراخوانی ابزار (`tools/call`)

متد `tools/call` است. نام ابزار و آرگومان‌ها در `params` قرار می‌گیرند:

</div>

In [41]:
r = send(
    {
        "jsonrpc": "2.0",
        "id": 3,
        "method": "tools/call",
        "params": {"name": "add", "arguments": {"a": 2, "b": 3}},
    },
    session_id=session_id,
    protocol_version=PROTOCOL,
)

>>> REQUEST
{
  "jsonrpc": "2.0",
  "id": 3,
  "method": "tools/call",
  "params": {
    "name": "add",
    "arguments": {
      "a": 2,
      "b": 3
    }
  }
}

<<< RESPONSE  (HTTP 200, content-type: text/event-stream)
event: message
data: {"jsonrpc":"2.0","id":3,"result":{"content":[{"type":"text","text":"5"}],"structuredContent":{"result":5},"isError":false}}




<div dir="rtl">

نتیجه دو شکل دارد:

- `content`: نتیجه به‌شکل **متن**، برای اینکه مدل زبانی بخواند.
- `structuredContent`: نتیجه به‌شکل **داده‌ی ساختاریافته** (`{"result": 5}`)، برای اینکه برنامه بتواند مستقیم از آن استفاده کند.

`id` پاسخ (`3`) با `id` درخواست یکی است. کلاینت با همین شماره می‌فهمد کدام پاسخ مال کدام درخواست است.

## دو نوع خطا

در MCP دو نوع خطای کاملاً متفاوت داریم.

**نوع ۱: خطای پروتکل.** درخواست از نظر JSON-RPC مشکل دارد، مثلاً متدی که وجود ندارد. پاسخ به‌جای `result` یک شیء `error` دارد:

</div>

In [43]:
r = send({"jsonrpc": "2.0", "id": 4, "method": "tools/run", "params": {}}, session_id=session_id, protocol_version=PROTOCOL)

>>> REQUEST
{
  "jsonrpc": "2.0",
  "id": 4,
  "method": "tools/run",
  "params": {}
}

<<< RESPONSE  (HTTP 200, content-type: text/event-stream)
event: message
data: {"jsonrpc":"2.0","id":4,"error":{"code":-32602,"message":"Invalid request parameters","data":""}}




<div dir="rtl">

> کد استاندارد JSON-RPC برای «متد پیدا نشد» عدد `-32601` است. این SDK پیام را قبل از هر چیز با لیست متدهای MCP اعتبارسنجی می‌کند و برای همین `-32602` («پارامترهای نامعتبر») برمی‌گرداند. کد دقیق به پیاده‌سازی بستگی دارد؛ مهم این است که پاسخ `error` دارد، نه `result`.

**نوع ۲: خطای اجرای ابزار.** درخواست درست است، ولی خود ابزار هنگام اجرا خطا داده است، مثلاً تقسیم بر صفر یا ورودی با نوع اشتباه. پاسخ یک `result` عادی است، ولی `isError: true` دارد و متن خطا در `content` آمده است:

</div>

In [45]:
r = send(
    {"jsonrpc": "2.0", "id": 5, "method": "tools/call", "params": {"name": "divide", "arguments": {"a": 1, "b": 0}}},
    session_id=session_id,
    protocol_version=PROTOCOL,
)
print()
r = send(
    {"jsonrpc": "2.0", "id": 6, "method": "tools/call", "params": {"name": "add", "arguments": {"a": "two", "b": 3}}},
    session_id=session_id,
    protocol_version=PROTOCOL,
)

>>> REQUEST
{
  "jsonrpc": "2.0",
  "id": 5,
  "method": "tools/call",
  "params": {
    "name": "divide",
    "arguments": {
      "a": 1,
      "b": 0
    }
  }
}

<<< RESPONSE  (HTTP 200, content-type: text/event-stream)
event: message
data: {"jsonrpc":"2.0","id":5,"result":{"content":[{"type":"text","text":"Error executing tool divide: float division by zero"}],"isError":true}}



>>> REQUEST
{
  "jsonrpc": "2.0",
  "id": 6,
  "method": "tools/call",
  "params": {
    "name": "add",
    "arguments": {
      "a": "two",
      "b": 3
    }
  }
}

<<< RESPONSE  (HTTP 200, content-type: text/event-stream)
event: message
data: {"jsonrpc":"2.0","id":6,"result":{"content":[{"type":"text","text":"Error executing tool add: 1 validation error for addArguments\na\n  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='two', input_type=str]\n    For further information visit https://errors.pydantic.dev/2.13/v/int_parsing"}],"isError":true}}




<div dir="rtl">

چرا این تفاوت مهم است؟

| | خطای پروتکل | خطای اجرای ابزار |
|---|---|---|
| شکل پاسخ | `error` | `result` با `isError: true` |
| چه کسی آن را می‌بیند؟ | برنامه‌ی کلاینت | **مدل زبانی** |
| نتیجه | برنامه باید مشکل را رفع کند | مدل متن خطا را می‌خواند و می‌تواند با ورودی درست دوباره تلاش کند |

## چرا Session مهم است؟

سرور ما **حالت‌دار (stateful)** است: هر کلاینت را با شناسه‌ی session اش می‌شناسد. اگر بدون `Mcp-Session-Id` درخواست بفرستیم، سرور آن را رد می‌کند:

</div>

In [48]:
r = send({"jsonrpc": "2.0", "id": 7, "method": "tools/list"})

>>> REQUEST
{
  "jsonrpc": "2.0",
  "id": 7,
  "method": "tools/list"
}

<<< RESPONSE  (HTTP 400, content-type: application/json)
{"jsonrpc":"2.0","id":"server-error","error":{"code":-32600,"message":"Bad Request: Missing session ID"}}


<div dir="rtl">

## گام ۵: پایان ارتباط

در Streamable HTTP، کلاینت با یک درخواست `DELETE` به همان آدرس، session را می‌بندد. بعد از آن، سرور آن شناسه را دیگر نمی‌شناسد:

</div>

In [50]:
r = httpx.delete(URL, headers={**BASE_HEADERS, "Mcp-Session-Id": session_id, "MCP-Protocol-Version": PROTOCOL})
print("DELETE ->", r.status_code)

r = send({"jsonrpc": "2.0", "id": 8, "method": "tools/list"}, session_id=session_id, protocol_version=PROTOCOL)

DELETE -> 200
>>> REQUEST
{
  "jsonrpc": "2.0",
  "id": 8,
  "method": "tools/list"
}

<<< RESPONSE  (HTTP 404, content-type: application/json)
{"jsonrpc":"2.0","id":"server-error","error":{"code":-32600,"message":"Not Found: Session has been terminated"}}


<div dir="rtl">

## جمع‌بندی

کل چرخه‌ای که دستی انجام دادیم:

| مرحله | پیام | `id` دارد؟ | پاسخ |
|---|---|---|---|
| ۱. راه‌اندازی | `initialize` | بله | قابلیت‌های سرور و `Mcp-Session-Id` |
| | `notifications/initialized` | خیر (Notification) | `202` بدون بدنه |
| ۲. تبادل پیام | `tools/list` | بله | لیست ابزارها و schema آن‌ها |
| | `tools/call` | بله | `content`، `structuredContent` و `isError` |
| ۳. پایان | `DELETE` (HTTP) | — | بسته شدن session |

### در نسخه‌ی جدید مشخصات MCP (2026-07-28) چه عوض شده؟

- `initialize`، `notifications/initialized` و هدر `Mcp-Session-Id` **حذف شده‌اند**. پروتکل بدون حالت است.
- هر درخواست، نسخه‌ی پروتکل و قابلیت‌های کلاینت را خودش در فیلد `_meta` حمل می‌کند.
- یک متد جدید `server/discover` اضافه شده تا کلاینت (در صورت نیاز) قابلیت‌های سرور را بپرسد.
- نتیجه: هر نسخه‌ی سرور پشت Load Balancer می‌تواند به هر درخواستی جواب دهد.

مفاهیمی که در این نوت‌بوک دیدیم (JSON-RPC، `tools/list`، `tools/call` و دو نوع خطا) در نسخه‌ی جدید هم دقیقاً همین‌طورند.

### برای کاربران لینوکس: همین کار با `curl`

فایل `script.sh` زیر همان گام‌ها را با `curl` انجام می‌دهد. روی ویندوز می‌توانید آن را در **Git Bash** یا **WSL** اجرا کنید.

</div>

```bash
#!/usr/bin/env bash
S=http://127.0.0.1:8000/mcp
ACCEPT='application/json, text/event-stream'
CT='application/json'

# 1) initialize  -> read the Mcp-Session-Id response header
SID=$(curl -sS -D - -o /dev/null -H "Accept: $ACCEPT" -H "Content-Type: $CT" -X POST $S \
  -d '{"jsonrpc":"2.0","id":1,"method":"initialize","params":{"protocolVersion":"2025-11-25","capabilities":{},"clientInfo":{"name":"bash","version":"1.0"}}}' \
  | sed -nE 's/^Mcp-Session-Id:[[:space:]]*//Ip' | tr -d '\r')
echo "SID=$SID"

# 2) notifications/initialized
curl -sS -H "Accept: $ACCEPT" -H "Content-Type: $CT" -H "Mcp-Session-Id: $SID" -X POST $S \
  -d '{"jsonrpc":"2.0","method":"notifications/initialized"}'

# 3) tools/call
curl -sS -H "Accept: $ACCEPT" -H "Content-Type: $CT" -H "Mcp-Session-Id: $SID" -X POST $S \
  -d '{"jsonrpc":"2.0","id":2,"method":"tools/call","params":{"name":"add","arguments":{"a":2,"b":3}}}'
```

<div dir="rtl">

در پایان سرور را خاموش می‌کنیم:

</div>

In [56]:
stop_server(server)

Server stopped (pid 3288)
